## Introducción

Este proyecto presenta la aplicación de diversas técnicas de minería de datos sobre tres conjuntos de datos pertenecientes a distintos dominios: agricultura, operaciones con drones y producción industrial. A través de estos casos de estudio se desarrolla un flujo de trabajo completo que abarca desde el preprocesamiento de los datos hasta la construcción, evaluación e interpretación de modelos predictivos.

A lo largo del notebook se emplean técnicas de regresión, clasificación, selección de variables, validación de modelos, predicción de series temporales y aprendizaje no supervisado, utilizando herramientas del ecosistema científico de Python.

El objetivo es mostrar cómo diferentes metodologías de minería de datos pueden adaptarse a problemas de naturaleza diversa, ilustrando tanto el proceso analítico como la interpretación de los resultados obtenidos.

### Contenido

- **Agricultura:** predicción del rendimiento de los cultivos y detección de plagas mediante técnicas de regresión y clasificación.
- **Operaciones con drones:** modelado del consumo energético por entrega a partir de variables operativas y ambientales.
- **Producción industrial:** análisis y predicción del Índice de Producción Industrial (IPI) mediante modelos de series temporales.


## Preliminares

En primer lugar reservamos un espacio para la configuración de la sesión que incluye la conexión al drive (o subida directa de archivos), carga de las funciones y datos.

In [ ]:
# Importar funciones
from NuestrasFunciones import *

In [ ]:
from IPython.display import Image, display

display(Image(filename="Tabla_Variables_Agri.png"))

In [ ]:
# Lectura datos agricultura
import pandas as pd

df_agri = pd.read_csv("Datos_agricultura_25.csv")
df_agri.head()

In [ ]:
# Lectura datos Ipi_Esp
df_ipi = pd.read_excel("IPI_Esp.xlsx")
df_ipi.head()

In [ ]:
# Lectura datos drones
df_drones = pd.read_excel("Datos_drones_25.xlsx")
df_drones.head()

## Limpieza de datos

In [ ]:
#Comprobación de datos

df_agri.info()
df_agri.describe()
df_agri.head()

In [ ]:
# Corrección de errores

# Eliminar columna correctamente
df_agri = df_agri.drop(columns=['Unnamed: 0'], errors='ignore')

# Eliminar identificador
df_agri = df_agri.drop(columns=['id'], errors='ignore')

Eliminamos la variable "Unnamed: 0" porque es simplemente un índice sin utilidad analítica. También se elimina la variable "id", ya que es un identificador único y no aporta información relevante para el modelo.

In [ ]:
# Cambios de tipos de variables

# Variables categóricas
df_agri['riego'] = df_agri['riego'].astype('category')
df_agri['tipo_suelo'] = df_agri['tipo_suelo'].astype('category')
df_agri['variedad_semilla'] = df_agri['variedad_semilla'].astype('category')
df_agri['uso_pesticidas'] = df_agri['uso_pesticidas'].astype('category')
df_agri['historial_plagas'] = df_agri['historial_plagas'].astype('category')
df_agri['plaga_severa'] = df_agri['plaga_severa'].astype('category')

In [ ]:
# Distribuciones de las variables corregidas

import matplotlib.pyplot as plt
import seaborn as sns

df_agri.hist(figsize=(12,8))
plt.show()

Se analizan las distribuciones de las variables después de corregir errores y ajustar los tipos de datos. Para ello, utilizamos histogramas de las variables numéricas con el objetivo de entender su comportamiento y detectar posibles valores atípicos.

In [ ]:
# Separar variables objetivo

y_cont = df_agri['rendimiento_kg_ha']
y_bin = df_agri['plaga_severa']

In [ ]:
# Crear input de predictores
X = df_agri.drop(columns=['rendimiento_kg_ha', 'plaga_severa'])

Se separan las variables objetivo del conjunto de datos. La variable continua es el rendimiento del cultivo (rendimiento_kg_ha) y la variable binaria es plaga_severa. El resto de variables se utilizan como predictores y forman la matriz X.

En este punto no se eliminan más variables (hemos eliminado el identificador y la variable "Unnamed:0" anteriormente), ya que todas pueden aportar información útil para el modelo.

In [ ]:
# Incidencia outliers

df_agri.select_dtypes(include=['float64','int64']).apply(lambda x: gestiona_outliers(x, 'check'))

In [ ]:
# Gestión efectiva de outliers

df_agri[df_agri.select_dtypes(include=['float64','int64']).columns] = \
df_agri.select_dtypes(include=['float64','int64']).apply(lambda x: gestiona_outliers(x, 'winsor'))

Se revisan los valores atípicos en las variables numéricas y, en general, no son muchas. Para tratarlos, utilizamos la winsorización, que consiste en limitar los valores extremos sin eliminar datos. Así se reduce su efecto sin perder información del conjunto de datos.

In [ ]:
# Unir variables categóricas

df_agri.select_dtypes(include='object').nunique()

In [ ]:
# Incidencia de missings
df_agri.isnull().sum()

In [ ]:
# Gestión efectiva de missings
# Tratamiento de valores faltantes
df_agri['lluvia_mm'] = df_agri['lluvia_mm'].fillna(df_agri['lluvia_mm'].mean())
df_agri['ph_suelo'] = df_agri['ph_suelo'].fillna(df_agri['ph_suelo'].mean())
df_agri['materia_organica_pct'] = df_agri['materia_organica_pct'].fillna(df_agri['materia_organica_pct'].mean())

df_agri['uso_pesticidas'] = df_agri['uso_pesticidas'].fillna(df_agri['uso_pesticidas'].mode()[0])

df_agri.isnull().sum()

El dataset presenta valores faltantes en varias variables. Se decide imputar las variables numéricas con la media, al tratarse de variables continuas, y la variable categórica con la moda, al ser el valor más frecuente.

Tras la imputación, comprobamos que no quedan valores faltantes en el conjunto de datos.

In [ ]:
# Datos depurados, análisis de cambios
df_agri.describe()


#### Observaciones:

Tras el proceso de depuración, obtenemos un conjunto de datos limpio y preparado para la modelización. Hemos tratado los valores atípicos mediante winsorización y los valores faltantes mediante imputación, sin necesidad de eliminar observaciones.

En general, los datos presentan valores coherentes y no se observan problemas relevantes que puedan afectar al análisis posterior.

## Generar variables aleatorias de control y presentar el ranking de variables

In [ ]:
# Generar variables aleatorias de control
import numpy as np

np.random.seed(0)

df_agri['random_1'] = np.random.normal(size=len(df_agri))
df_agri['random_2'] = np.random.uniform(size=len(df_agri))

# Discretizar la variable objetivo continua
df_agri['rendimiento_cat'] = pd.qcut(df_agri['rendimiento_kg_ha'], q=4, labels=False)

In [ ]:
# Ranking V de Cramer vs. obj continua
import pandas as pd
import scipy.stats as stats

def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tabla)[0]
    n = tabla.sum().sum()
    r, k = tabla.shape
    return np.sqrt(chi2 / (n * (min(k-1, r-1))))

# Convertir variables numéricas en categóricas (bins)
df_cramer = df_agri.copy()

for col in df_cramer.select_dtypes(include=['float64', 'int64']).columns:
    if col not in ['rendimiento_kg_ha']:
        df_cramer[col] = pd.qcut(df_cramer[col], q=4, duplicates='drop')

# Calcular V de Cramer
resultados = {}

for col in df_cramer.columns:
    if col not in ['rendimiento_kg_ha', 'rendimiento_cat']:
        resultados[col] = cramers_v(df_cramer[col], df_cramer['rendimiento_cat'])

# Ranking
ranking = pd.Series(resultados).sort_values(ascending=False)
ranking

Se generan dos variables aleatorias de control que, en principio, no deberían tener relación con la variable objetivo.

Para poder aplicar la V de Cramer, se discretiza la variable objetivo continua en varios grupos y se hace lo mismo con las variables numéricas. Así se puede medir la relación entre variables. A partir del ranking, se observa que las variables más relacionadas con el rendimiento son principalmente lluvia_mm, plaga_severa y riego. También aparecen con cierta importancia fertilizante_kg_ha y temp_media_c. Por otro lado, las variables aleatorias presentan valores muy bajos, lo cual tiene sentido porque no deberían aportar información. Esto confirma que el método está funcionando correctamente.

En conclusión, las variables que parecen más relevantes para el modelo son las que presentan mayor V de Cramer, especialmente lluvia_mm y plaga_severa. Por el contrario, aquellas variables cuyos valores son similares a los de las variables aleatorias tendrían menor capacidad explicativa y podrían no ser relevantes para el modelo.


## Ajustar el modelo completo de referencia para la predicción de la variable objetivo continua.

In [ ]:
#Fórmula modelo completo
cols = df_agri.columns.drop('rendimiento_kg_ha')
formula = 'rendimiento_kg_ha ~ ' + ' + '.join(cols)

In [ ]:
# Ajuste modelo completo
import statsmodels.formula.api as smf

modelo = smf.ols(formula, data=df_agri).fit()

# Summary modelo completo
modelo.summary()


####Resultados:

- El modelo tiene un R² de 0.903. Esto significa que aproximadamente el 90% de la variación del rendimiento del cultivo se explica con las variables que hemos incluido. Es un valor bastante alto, así que el modelo funciona bien y recoge bastante bien la información de los datos.
- La hipótesis nula dice que ninguna de las variables tiene efecto sobre el rendimiento. En este caso, el p-valor del modelo es prácticamente 0, así que rechazamos esa hipótesis. Esto significa que el modelo en conjunto sí es útil y que hay variables que influyen en el rendimiento.
- El modelo incluye 24 efectos (sin contar la constante). Existen bastantes variables (12) que son significativos al 5%, lo que indica que sí hay varias variables que influyen en el rendimiento. Por ejemplo, riego, tipo de suelo, lluvia, temperatura, fertilizante y materia orgánica. Además, las variables aleatorias que hemos añadido no son significativas, lo cual es buena señal porque confirma que el modelo no está captando relaciones por casualidad.
- En general no se ven problemas importantes. Están bastante centrados y no parecen seguir un patrón raro, lo que indica que el modelo está bien ajustado. Aunque no siguen perfectamente una distribución normal, esto no parece afectar demasiado al resultado, así que el modelo se puede considerar válido.

## Aplicar un modelo Lasso para la selección automática de variables.

In [ ]:
# Selección automática Lasso

from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

X = df_agri.drop(columns=['rendimiento_kg_ha', 'plaga_severa']) #Antes de aplicar Lasso, es necesario asegurar que no existen valores faltantes en las variables explicativas, ya que este modelo no los admite. Aunque previamente se han tratado los missings, algunas transformaciones pueden haber generado nuevos valores perdidos, por lo que se eliminan o imputan antes del ajuste.
X_lasso = X.select_dtypes(include=['float64', 'int64'])
y = y_cont

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_lasso)

lasso = LassoCV(cv=5, random_state=0)
lasso.fit(X_scaled, y)


In [ ]:
# Variables incluidas
coef_lasso = pd.Series(lasso.coef_, index=X_lasso.columns)

vars_lasso = coef_lasso[coef_lasso != 0]
vars_lasso

In [ ]:
# Ajuste del modelo con statsmodels

formula_lasso = 'rendimiento_kg_ha ~ lluvia_mm + temp_media_c + fertilizante_kg_ha + ph_suelo + materia_organica_pct + mano_obra_horas + riego + tipo_suelo + variedad_semilla + uso_pesticidas + historial_plagas'

import statsmodels.formula.api as smf
modelo_lasso = smf.ols(formula_lasso, data=df_agri).fit()

In [ ]:
#Summary modelo Lasso
modelo_lasso.summary()

#### Resultados:

- El modelo Lasso tiene un R² de 0.833. Este valor es algo menor que el del modelo completo (0.903), lo que indica que se pierde algo de capacidad explicativa al reducir el número de variables. Aun así, el modelo sigue funcionando bastante bien y es más sencillo, por lo que puede ser una buena opción si buscamos un modelo más simple.
- El modelo incluye 18 efectos. De ellos, varios son significativos al 5%, lo que indica que las variables seleccionadas por Lasso sí tienen una influencia real sobre el rendimiento. Exactamente son 9, entre las más importantes destacan riego, tipo de suelo, lluvia, temperatura, fertilizante y materia orgánica.


## Agregar las mejores transformaciones de las variables continuas y aplicar un selección secuencial de variables.

In [ ]:
# Generar transformaciones

import numpy as np

X_transf = X.copy()

# Ajuste para evitar log de negativos
min_lluvia = X_transf['lluvia_mm'].min()

# Log transformaciones
X_transf['log_lluvia'] = np.log(X_transf['lluvia_mm'] - min_lluvia + 1)
X_transf['log_fertilizante'] = np.log(X_transf['fertilizante_kg_ha'] + 1)
X_transf['log_materia_organica'] = np.log(X_transf['materia_organica_pct'] + 1)

# Raíz cuadrada
X_transf['sqrt_area'] = np.sqrt(X_transf['area_ha'])
X_transf['sqrt_mano_obra'] = np.sqrt(X_transf['mano_obra_horas'])

In [ ]:
# Unir al input de predictores

X_transf.head()

Nota: Este notebook requiere la librería mlxtend para la selección secuencial de variables.

In [ ]:
# Aplicar SFS
import sys
!{sys.executable} -m pip install mlxtend

In [ ]:
# Aplicar SFS
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Dummies para variables categóricas
X_sfs = pd.get_dummies(X_transf, drop_first=True)

# Escalar variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sfs)

# Modelo base
lr = LinearRegression()

# SFS backward parsimonious
sfs = SFS(
    lr,
    k_features='parsimonious',
    forward=False,
    floating=False,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

sfs.fit(X_scaled, y_cont)

In [ ]:
# Variables selccionadas
features_idx = list(sfs.k_feature_idx_)
features_sfs = X_sfs.columns[features_idx]

In [ ]:
# Ajuste con statsmodels del modelo sfs transformaciones
import statsmodels.formula.api as smf

# Crear fórmula
features_formula = [f'Q("{col}")' for col in features_sfs]
formula_sfs = 'rendimiento_kg_ha ~ ' + ' + '.join(features_formula)

# Ajustar modelo
modelo_sfs = smf.ols(formula_sfs, data=X_sfs.join(y_cont)).fit()

In [ ]:
modelo_sfs.summary()

#### Resultados:

- Se observan algunas transformaciones interesantes, sobre todo log_lluvia y log_fertilizante. Tiene bastante sentido porque la relación con el rendimiento no tiene por qué ser lineal, y con el log se captura mejor ese efecto.
- El modelo tiene un R² de 0.852. Es menor que el del modelo completo (0.903), pero mejora respecto al Lasso (0.833). En general, se pierde un poco de ajuste, pero a cambio el modelo es más simple y sigue explicando bastante bien el rendimiento.
- Las variables transformadas son significativas. Tanto log_lluvia como log_fertilizante tienen p-valores prácticamente 0, así que aportan información relevante al modelo.
- En cuanto a los residuos, no hay una mejora muy clara. Siguen sin ser perfectamente normales, pero tampoco es algo grave. En general, el modelo sigue estando bien ajustado y es válido.

## Ajustar el modelo completo para la predicción de la variable objetivo binaria.




In [ ]:
# Fórmula completa objetivo binaria
cols = df_agri.columns.drop('plaga_severa')
formula_bin = 'plaga_severa ~ ' + ' + '.join(cols)

In [ ]:
# Ajuste modelo bin

import statsmodels.formula.api as smf

# Convertir variable objetivo a numérica (0/1)
df_agri['plaga_severa'] = df_agri['plaga_severa'].astype(int)

# Ajustar modelo
modelo_bin = smf.logit(formula_bin, data=df_agri).fit()

In [ ]:
# Summmary modelo completo bin
modelo_bin.summary()

#### Resultados:

- La métrica de ajuste es el pseudo R², que en este caso es aproximadamente 0.115. Es un valor bastante bajo, así que el modelo no explica muy bien la probabilidad de que haya plaga severa. Aun así, en este tipo de modelos es bastante normal, pero sí indica que el ajuste es limitado.
- El modelo se podría mejorar bastante. Hay muchas variables que no son significativas, así que tendría sentido hacer una selección de variables. También se podrían probar transformaciones o incluir interacciones. Además, sería interesante evaluar el modelo con otras métricas como accuracy o ROC para ver mejor cómo está funcionando en la práctica.


## Reducir el modelo eliminando los efectos no significativos hasta obtener el mejor modelo (más simple y con capacidad predictiva similar al completo).



In [ ]:
# Formula reducida
formula_red = 'plaga_severa ~ tipo_suelo + uso_pesticidas + historial_plagas + lluvia_mm + temp_media_c + ph_suelo + materia_organica_pct'


In [ ]:
# Ajuste modelo reducido
import statsmodels.formula.api as smf

modelo_red = smf.logit(formula_red, data=df_agri).fit()

In [ ]:
# Summary modelo reducido
modelo_red.summary()

#### Resultados:

- En el modelo reducido se quedan bastantes menos variables, 12 variables, frente a todas las que tenía el modelo completo. El ajuste prácticamente no cambia, ya que el R² se mantiene bastante parecido. Así que compensa, porque tenemos un modelo mucho más simple sin perder casi capacidad explicativa.



## Comparar con cross_val_lin() los modelos completo y reducido de los apartados anteriores.



In [ ]:
# Lista de fórmulas
formulas = [
    formula_bin,   # modelo completo
    formula_red    # modelo reducido
]

In [ ]:
# Aplicar cros val a la lista
# Modelo completo
cv_completo = cross_val_lin(formula_bin, df_agri)

# Modelo reducido
cv_reducido = cross_val_lin(formula_red, df_agri)

In [ ]:
# Extraer el dataset y darle formato
import pandas as pd

df_cv = pd.DataFrame({
    'Completo': cv_completo,
    'Reducido': cv_reducido
})

In [ ]:
# Graficar boxplots
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=df_cv)
plt.show()

#### Resultados:

- En los boxplots se ve que ambos modelos tienen resultados bastante parecidos, pero el modelo reducido tiene una media ligeramente mayor y una dispersión similar o incluso algo menor. Esto indica que el modelo reducido generaliza igual o incluso mejor que el completo. Por tanto, tiene sentido quedarse con el modelo reducido, ya que es más simple y funciona prácticamente igual o mejor.


## Series temporales

Los datos a utilizar en esta parte son los contenidos en el conjunto **IPI_Esp.xlsx**

## Convertir el conjunto de datos a una serie temporal comprensible para python.



In [ ]:
# Conversión a serie temporal
# Limpiar espacios
df_ipi['Date'] = df_ipi['Date'].str.strip()

# Convertir a datetime (reemplazando M por -)
df_ipi['Date'] = pd.to_datetime(df_ipi['Date'].str.replace('M', '-'), format='%Y-%m')

# Poner como índice
df_ipi = df_ipi.set_index('Date')

# Serie
serie = df_ipi['IPI Nacional']
serie.head()


In [ ]:
# Componentes
from statsmodels.tsa.seasonal import seasonal_decompose

descomp = seasonal_decompose(serie, model='additive', period=12)
descomp.plot()

In [ ]:
# Prueba de estacionariedad
# Prueba de estacionariedad
from statsmodels.tsa.stattools import adfuller

adf_test = adfuller(serie)

adf_test

#### Resultados:

- En la descomposición se ve bastante claro que la serie tiene una tendencia creciente con el tiempo, aunque hay algún cambio más brusco en ciertos periodos. También hay una estacionalidad muy marcada, porque los patrones se repiten prácticamente igual cada año. Los residuos parecen bastante aleatorios, aunque en algunos momentos tienen algo más de variación.
- No diría que es una serie estacionaria. Se ve que la media va cambiando con el tiempo por la tendencia y además hay estacionalidad. Esto también lo confirma el test ADF, que da un p-valor bastante alto (alrededor de 0.39), así que no se puede rechazar que la serie no sea estacionaria. Por eso, habría que transformarla antes de modelarla.


## Ajustar el modelo de suavizado exponencial más adecuado para el caso.

In [ ]:
# Modelo de suavizado adecuado

from statsmodels.tsa.holtwinters import ExponentialSmoothing

serie.index.freq = 'MS'

modelo_hw = ExponentialSmoothing(
    serie,
    trend='add',
    seasonal='add',
    seasonal_periods=12
).fit()
pred = modelo_hw.fittedvalues

In [ ]:
# Evaluación del modelo de suavizado

import matplotlib.pyplot as plt

# Ajuste vs real
plt.figure(figsize=(10,5))
plt.plot(serie, label='Real')
plt.plot(pred, label='Ajuste', color='red')
plt.legend()
plt.show()

# Residuos
residuos = serie - pred

plt.figure(figsize=(10,4))
plt.plot(residuos)
plt.title("Residuos")
plt.show()

#### Resultados:

- El modelo más adecuado es Holt-Winters con tendencia y estacionalidad aditiva, porque la serie tiene claramente ambos componentes. El ajuste es bastante bueno, ya que la línea ajustada sigue bien a la serie real y recoge tanto la tendencia como la estacionalidad. Aun así, en algunos periodos con cambios más bruscos no ajusta perfectamente
- En los modelos de series temporales es importante que los residuos no tengan ningún patrón y se comporten de forma aleatoria. En este caso, en general los residuos parecen bastante aleatorios, aunque en algunos momentos tienen más variación, así que no se cumple del todo, pero el modelo es bastante razonable.


## Ajustar el modelo Sarimax que la función autoarima considere más adecuado.



Nota: Para poder ajustar el modelo SARIMAX, se ha instalado la librería pmdarima.

In [ ]:
# Modelo de autoarima
import sys
!{sys.executable} -m pip install pmdarima

In [ ]:
# Modelo de autoarima

from pmdarima import auto_arima

modelo_arima = auto_arima(
    serie,
    seasonal=True,
    m=12,
    trace=True,
    error_action='ignore',
    suppress_warnings=True
)

modelo_arima.summary()

In [ ]:
# Evaluación del modelo de autoarima

# Predicción ajustada
pred_arima = modelo_arima.predict_in_sample()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(serie, label='Real')
plt.plot(pred_arima, label='Ajuste', color='red')
plt.legend()
plt.show()

# Residuos
residuos_arima = serie - pred_arima

plt.figure(figsize=(10,4))
plt.plot(residuos_arima)
plt.title("Residuos")
plt.show()

#### Resultados:

- El modelo seleccionado por autoarima es un SARIMAX con componentes autorregresivos, de medias móviles y estacionales. El ajuste es bastante bueno, ya que la predicción sigue muy de cerca la serie real y capta bien tanto la tendencia como la estacionalidad.
- En general, la mayoría de los parámetros son significativos, ya que tienen p-valores muy bajos (cercanos a 0). Aunque alguno no lo es, la mayoría sí aportan información relevante al modelo.
- Se observa una ligera mejora respecto al modelo de suavizado exponencial, ya que el SARIMAX ajusta mejor algunos cambios y tiene en cuenta la dependencia temporal de forma más completa.
- El test de Ljung-Box contrasta si los residuos están autocorrelacionados. La hipótesis nula es que no hay autocorrelación. En este caso, como el p-valor es alto (≈ 0.98), no se rechaza la hipótesis nula, por lo que los residuos se pueden considerar independientes y el modelo es adecuado.


## Técnicas no supervisadas

Los datos a utilizar en esta parte son los contenidos en el conjunto **Datos_drones_25.xlsx**

In [ ]:
# Mostrar tabla de variables
from IPython.display import Image, display

display(Image(filename="Tabla_drones.png", width=600))

## Presentar la matriz de correlaciones entre las variables del conjunto de datos.

In [ ]:
# Matriz de correlaciones

import seaborn as sns
import matplotlib.pyplot as plt

corr = df_drones.corr(numeric_only=True)

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='RdBu_r')
plt.title("Matriz de correlaciones")
plt.show()

#### Resultados:

- Se intuye un problema de multicolinealidad, ya que hay variables muy correlacionadas entre sí (por ejemplo, distancia y desnivel, o viento medio y ráfagas). Esto ocurre cuando varias variables explicativas aportan información muy similar. Como consecuencia, los coeficientes del modelo pueden volverse inestables, difíciles de interpretar y el modelo puede perder capacidad predictiva.
- Este problema se puede abordar utilizando técnicas no supervisadas como el PCA. El PCA permite transformar las variables originales en nuevas variables (componentes) que no están correlacionadas entre sí. De esta forma, se elimina la información redundante que hay entre variables muy relacionadas. Además, se reduce el número de variables manteniendo la mayor parte de la información, lo que ayuda a obtener un modelo más estable y más fácil de interpretar.


## Calcular el KMO de la muestra para todas las variables excepto la objetivo (y si existen identificadores o variables rechazadas).


Nota: Se instala la librería factor_analyzer para poder calcular el estadístico KMO, ya que no estaba disponible en el entorno de trabajo.

In [ ]:
# KMO
import sys
!{sys.executable} -m pip install factor_analyzer

In [ ]:
# KMO
from factor_analyzer.factor_analyzer import calculate_kmo

# Quitamos la variable objetivo y posibles identificadores
X = df_drones.drop(columns=['consumo_energia_por_entrega', 'Unnamed: 0'])

kmo_all, kmo_model = calculate_kmo(X)

kmo_model


In [ ]:
# Ajuste modelo reducción

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Escalar datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA completo
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# Gráfico

import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('Varianza explicada acumulada (PCA)')
plt.grid()
plt.show()

#### Resultados:

- El valor del KMO es aproximadamente 0.79, lo que indica una buena adecuación de la muestra. Esto significa que las variables están suficientemente correlacionadas y que aplicar técnicas de reducción de dimensionalidad como el PCA es apropiado en este caso.
- Es adecuado reducir la dimensionalidad del conjunto de predictores, ya que además existe multicolinealidad entre variables. La reducción nos permite eliminar redundancias y simplificar el modelo sin perder demasiada información.
- A partir del gráfico de varianza explicada acumulada, se observa que con 3 componentes ya se explica aproximadamente el 85-90% de la varianza total. Por tanto, se pueden retener 3 componentes como una solución equilibrada entre simplicidad y capacidad explicativa.
- Las componentes principales pueden interpretarse como combinaciones de variables relacionadas. Por ejemplo, una componente puede estar asociada a las condiciones meteorológicas (viento y ráfagas), otra a las características de la ruta (distancia y desnivel), y otra a aspectos operativos (carga, tiempo de entrega). Esto nos permite resumir la información en factores más interpretable y reducir la complejidad del problema.


## Aplicar un modelo de clustering particional y decidir el número de grupos que resultan más adecuados.



In [ ]:
# Rescatar los scores del PCA para el número de componentes seleccionadas
X_pca_red = X_pca[:, :3]

In [ ]:
# Exploración número de clusters k-means

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

silhouette = {}

for k in range(2,7):
    kmeans = KMeans(n_clusters=k, random_state=0)
    labels = kmeans.fit_predict(X_pca_red)
    silhouette[k] = silhouette_score(X_pca_red, labels)

silhouette

In [ ]:
# Modelo elegido
kmeans = KMeans(n_clusters=3, random_state=0)  # cambia si te da otro mejor
labels = kmeans.fit_predict(X_pca_red)

In [ ]:
# Centroides
centroides = kmeans.cluster_centers_
centroides

In [ ]:
# Biplot por grupos

import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(X_pca_red[:,0], X_pca_red[:,1], c=labels, cmap='viridis')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')
plt.title('Clusters en espacio PCA')
plt.colorbar()
plt.show()

#### Resultados:

- La silueta toma un valor aproximado de 0.365, lo que indica una separación razonable entre los grupos (aunque no perfecta). La solución en dimensión reducida conserva alrededor del 90% de la variabilidad total, por lo que se mantiene la mayor parte de la información original.
- Los centroides de los clusters son:
Cluster 1: (-1.78, 1.25, 0.05)
Cluster 2: (2.46, 0.51, -0.06)
Cluster 3: (-0.51, -1.73, 0.01)
Estos valores representan la media de cada componente principal en cada grupo.
- En el gráfico biplot se observa una separación bastante clara entre los tres grupos en el espacio de las dos primeras componentes principales, lo que indica que el clustering es consistente.
- Los grupos se pueden caracterizar a partir de las componentes principales: un cluster presenta valores altos en la componente 1 (relacionada con variables operativas como carga o tiempo), otro destaca en la componente 2 (más asociada a condiciones externas como viento o temperatura), y un tercer grupo presenta valores más bajos en ambas, representando un perfil intermedio o diferente.
